# HPC Cluster Concepts

## 1. HPC Cluster Overview

<img src="./img/general_cluster_architecture.png"/>

A **cluster** is a collection of compute nodes. A **partition** is a logical grouping of nodes or resources. It is used to organize access to different types of hardware, such as CPU nodes, GPU nodes, or nodes with special characteristics.

A **node** is one compute server. Each node can contain one or more CPU sockets. A socket is the physical CPU package installed in the server.

Each socket contains multiple **CPU cores**. A CPU core is a physical computing unit. Depending on the processor and configuration, each core may expose one or more **CPU hardware threads**, typically two when simultaneous multithreading or hyperthreading is enabled. If this is disabled, one hardware thread usually corresponds to one physical core.

Modern processors are often divided into **NUMA domains**, also called locality domains. A NUMA domain is a group of CPUs that share memory that is physically closer to them. Accessing local memory is faster than accessing memory attached to another NUMA domain. A single socket can therefore contain one or more NUMA domains, depending on the processor architecture and BIOS configuration.

Some nodes may also contain **GPUs**, which are accelerator processors. GPUs can have locality relationships with specific CPU sockets or NUMA domains, so their placement in the hardware topology can matter for performance.

A **task** is a running process, for example an *MPI rank*. **Affinity** describes where that process is allowed to run: on which CPU, core, socket, or NUMA domain. Correct affinity helps keep processes close to the CPU cores, memory, and accelerators they use.

A simplified hierarchy is:

```shell
Cluster
  └── Partition
       └── Node
            └── Socket
                 └── NUMA / Locality Domain
                      └── CPU Core
                           └── CPU Hardware Thread
```

The key message is that performance is not only about how many CPUs a system has, but also about **where those CPUs, memory, tasks, and accelerators are located relative to each other.**

## 2. Slurm Cluster Overview

**Slurm** is the workload management layer that connects users to the compute infrastructure. In other words, **the HPC cluster provides the resources, while Slurm manages access to them**.
It queues jobs, allocates resources, launches workloads on compute nodes, and records usage for accounting and reporting.

<img src="./img/general_slurm_architecture.png"/>

An **HPC cluster** provides the physical and shared resources: login nodes, compute nodes, CPUs, memory, GPUs, high-speed network, and often **shared storage**. Users normally access the cluster through login nodes or other entry points, but heavy workloads should not run directly there. Instead, users submit jobs to Slurm using commands such as `sbatch` or `srun`.

Slurm receives these job requests, places them in a queue, and decides when and where they can run based on available resources, partitions, priorities, limits, and site policies. The Slurm controller, **slurmctld**, is responsible for scheduling and launching jobs on the selected compute nodes.

Once resources are available, Slurm starts the job on the compute nodes allocated to it. These nodes provide the CPUs, memory, and GPUs requested by the job.

Slurm also optionally includes an accounting component called **slurmdbd**. The **slurmdbd** service collects and stores job accounting information in a database. This includes details such as who ran a job, when it started and ended, which resources were allocated, how long the job ran, and how much CPU, memory, or GPU time was consumed. This information is used for reporting, fair-share calculations, usage analysis, debugging, and long-term accounting.

## 3. Merlin7 Slurm architecture overview

### The General HPC Slurm cluster

At **Paul Scherrer Institut**, **Merlin** is a series of centrally managed high-performance computing clusters, and is providing PSI staff and collaborators with scalable CPU and GPU resources for simulations, data analysis, and other data-intensive scientific workloads. This is operated by the [HPCE team](https://www.psi.ch/en/awi/hpce-group), in the [Scientific Computing Division](https://www.psi.ch/en/csd). PSI also run other Slurm clusters

**Merlin7** is the newest generation of the Merlin HPC environment. The cluster is based on Slurm, and is hosted on CSCS's Alps infrastructure in Lugano as an independent vCluster, and provides PSI users with flexible access to CPU and modern GPU resources for demanding scientific computing workloads.

In addition to Merlin, PSI also operates more specialized computing clusters for specific facilities, projects, or departments, such as the **Ra** data analysis cluster and the **SwissFEL** online/near-time computing infrastructure, which are tailored to experimental data acquisition, processing, and analysis workflows.

<img src="./img/merlin7_architecture.png"/>

Merlin7 is composed of **two Slurm clusters** that share the same accounting layer and are accessed through login or service entry points.

Users can access Merlin7 and submit jobs through several **entry points**:

- **`login001.merlin7.psi.ch` / `login002.merlin7.psi.ch`** are the interactive login nodes used by users to prepare and submit jobs. Access is possible through SSH protocol as well as NoMachine (desktop based). Typical commands for submitting jobs from the login nodes are: `sbatch`, `salloc`, `srun`.
- **`merlin7-jupyter.psi.ch` / `ondemand.psi.ch`** are web-based entry points which provide browser-based access to cluster services.
- **`service03`** is a non-interactive service node, used for application-driven job submissions

Merlin7 has two separate Slurm control planes:

* The **`merlin7`** Slurm cluster manages the CPU resources. Is managed by **2 `slurmctld` controllers** in a high-availability setup (**active/passive**) which are in charge of scheduling jobs to the Merlin7 CPU nodes
* The **`gmerlin7`** Slurm cluster manages the GPU resources. Is managed by **2 `slurmctld` controllers** in a high-availability setup (**active/passive**) which are in charge of scheduling jobs to the Merlin7 GPU nodes

Both Slurm clusters share the same accounting infrastructure. This layer consists of **2 `slurmdbd` services** in a high-availability setup (**active/passive**), responsible for collecting and exposing accounting information from both Slurm clusters, as well as managing Slurm users, accounts, and associations. The `slurmdbd` services are backed by a **MariaDB** database, which stores user and account information, job history, resource usage, and accounting records. This shared accounting layer allows both `merlin7` and `gmerlin7` to report usage consistently through the same backend.

The compute resources are divided into CPU and GPU systems:

* Merlin7 CPU nodes are AMD-based compute nodes and are exclusively used for CPU workloads. These nodes are managed by the `merlin7` Slurm cluster.
* Merlin7 GPU nodes are a mix of A100 (AMD-based) and Grace-Hopper (ARM-based) nodes, which are exclusively used for GPU workloads. These nodes are managed by the `gmerlin7` Slurm cluster.

The HPC system (login and compute nodes, and storage) is connected through the **Slingshot interconnect / system network backbone**. 

The shared storage layer is provided by and **HPC Lustre Storage**, which is a shared parallel filesystem accessible from the cluster components. This storage is used for user data, project data and shared scratch.

### Hands-on: Accessing the Merlin7 cluster

**Duration:** 10 minutes.

Access to the Merlin7 login nodes using your preferred protocol. For this, please use the PSI username (`psicourse-stud[01-50]`) and password which was provided to you. Possible options:

* Using SSH, as described [here](https://hpce.pages.psi.ch/merlin7/01-Quick-Start-Guide/accessing-interactive-nodes/#ssh-access)
* Using NoMachine, as described [here](https://hpce.pages.psi.ch/merlin7/02-How-To-Use-Merlin/nomachine/)
* Using Open OnDemand, as described [here](https://hpce.pages.psi.ch/merlin7/05-Open-OnDemand/open-ondemand/)


### Hands-on: Understanding hardware topology, NUMA, cores, threads, and affinity on Merlin7 nodes

**Duration:** 10 minutes.

Understand the basic hardware hierarchy of the nodes.

On the login node, run:

```bash
lscpu
```

Then focus on the most relevant fields:

```bash
lscpu | egrep 'CPU\(s\)|Thread|Core|Socket|NUMA|Model name'
```

Now run the same command on a CPU compute node, as follows:

```bash
srun --reservation psicourse01 \
  bash -lc "lscpu | egrep 'CPU\(s\)|Thread|Core|Socket|NUMA|Model name'"
```

Finally, run the same command on a GPU compute node, as follows:

```bash
srun -M gmerlin7 --reservation psicourse01 \
  bash -lc "lscpu | egrep 'CPU\(s\)|Thread|Core|Socket|NUMA|Model name'"
```

##### Questions

> **[INFO]** Edit this cell and write your answers below.

1. **How many sockets do the nodes have?**
> YOUR ANSWER HERE

3. **How many physical cores per socket?**
> YOUR ANSWER HERE

4. **How many hardware threads per core?**
> YOUR ANSWER HERE

5. **How many total hardware threads?**
> YOUR ANSWER HERE

6. **How many NUMA domains? Why do the CPU login and compute nodes differ if the processor is the same?**
> YOUR ANSWER HERE


## 4. Basic Slurm command workflow

We can divide common Slurm commands into four parts of the job lifecycle:
1. submitting or launching work,
2. monitoring the queue,
3. controlling jobs,
4. and checking accounting or efficiency information.

<img src="./img/slurm_basic.png"/>

The typical Slurm workflow starts with submitting or launching work through `sbatch`, `srun`, or `salloc`. Users then monitor their jobs and the cluster state with `squeue`, `sinfo`, and `sprio`.

If needed, jobs can be cancelled or inspected in more detail with `scancel` and `scontrol`. After completion, `sacct`, `sacctmgr`, and `seff` help users understand accounting information, limits, usage, and job efficiency.

### Submit and launch jobs

Users normally submit work to Slurm instead of running heavy workloads directly on login nodes. For batch workloads, `sbatch` is used to submit a job script to the scheduler. This is the standard approach for production jobs, longer calculations, and workflows that do not require direct interaction.

For interactive work, users can use `srun` or `salloc`. These commands request resources from Slurm and provide an interactive environment once the allocation is available. This is useful for testing, debugging, compiling, short exploratory runs, or preparing job scripts before submitting larger batch jobs.

The important idea is that users request compute resources from Slurm, and Slurm decides where and when the workload can run.

### Monitor jobs and cluster state

After submitting a job, users can inspect its status with `squeue`. This shows whether jobs are running, pending, or waiting for resources or policy conditions.

The `sinfo` command gives an overview of partitions and node states. It helps users understand which parts of the cluster are available and whether nodes are idle, allocated, drained, or unavailable.

The `sprio` command provides information about job priority. It can help explain why a pending job is behind another job in the queue, especially when fair-share, job age, partition, or other scheduling factors are involved.

Together, these commands help users answer basic questions such as whether their job is pending, which partitions are available, and why another job may start earlier.

### Control jobs

Users can cancel jobs with `scancel`. This is useful when a job was submitted with wrong parameters, is no longer needed, or is not behaving as expected.

For more detailed inspection and control, `scontrol` provides access to Slurm’s internal view of jobs, nodes, and partitions. It can be used to inspect detailed job information, hold or release jobs, and, for administrators, modify selected job or cluster parameters.

Some `scontrol` actions are available to normal users for their own jobs, while others are restricted to administrators.

### Account and tune jobs

After jobs have run, users can inspect accounting information with `sacct`. This provides job history and resource usage information, including completed, failed, cancelled, or running jobs, depending on site configuration.

The `sacctmgr` command is used for Slurm accounting management. It exposes information about accounts, users, associations, fair-share, QoS, and limits. This is especially useful for administrators, but it can also help explain scheduling behavior and resource limits.

For completed batch jobs, `seff` gives a simple efficiency summary. It helps users understand whether the requested CPUs, memory, and walltime matched the actual job usage. This information is useful for tuning future submissions and improving cluster efficiency.


# Hands-on: Understanding Slurm resource requests and resource use

**Estimated duration:** 1h / Until the end of the session.

## Purpose

This session is about **Slurm resource allocation**, not MPI or OpenMP programming. The supplied `mpi_omp_affinity` program is used as an inspection tool: it creates one process per Slurm task, starts a configurable number of threads in each task, allocates some test memory, and reports where those resources are actually placed.

Participants do not need to understand the C implementation. The important questions are:

- What did the job request from Slurm?
- What did Slurm allocate?
- How did `srun` launch work inside that allocation?
- How were CPUs divided among tasks?
- How did memory requests change with `--mem` and `--mem-per-cpu`?
- How can the allocation and actual use be checked during and after the job?

## Learning objectives

After the session, participants should be able to:

- explain the difference between a **job allocation** and a **job step**;
- interpret `--nodes`, `--ntasks`, `--ntasks-per-node`, and `--cpus-per-task`;
- calculate the total number of allocated CPUs;
- distinguish tasks from CPUs and threads;
- explain why `--cpus-per-task` does not itself create threads;
- understand the difference between `--mem` and `--mem-per-cpu`;
- inspect pending, running, and completed jobs with `squeue`, `scontrol`, `sstat`, and `sacct`;
- verify task and CPU placement using Slurm and the supplied inspection program;
- recognize underuse, oversubscription, and inconsistent resource requests.

## Preparation 

### Files

The following files can be found under the **[`./mpi_omp_affinity_hands_on`](./mpi_omp_affinity_hands_on/)** directory

```text
mpi_omp_affinity.c
Makefile
01_single_task.batch
02_tasks_and_cpus.batch
03_nodes_and_distribution.batch
04_job_steps.batch
05_memory_requests.batch
06_resource_mismatch.batch
07_memory_binding.batch
```

The executable accepts one optional argument: the number of MiB allocated **by each task**:

```bash
./mpi_omp_affinity 64
```

This application allocation is deliberately separate from the Slurm memory request. It lets us compare requested memory with memory actually touched by the workload.

### Software building

On the login node, load the necessary dependencies to compile the software. This is done through the **PSI Environment Modules**, which is a system that dynamically configures your shell environment so you can easily load, switch, and unload different software packages and versions.

```bash
module purge
module load gcc/14.3.0 mpich/5.0.1 hwloc/2.12.0

module list
```

Once the modules are loaded, compile the software as follows:

```bash
make clean
make
```

The compilation is only preparation. The remaining exercises focus on submitting and inspecting Slurm jobs.

***Do not use the login node for the resource-placement experiments.***

### Inspect the cluster before submitting jobs

Before running the software, start by checking where you are:

```bash
hostname
whoami
pwd
```

You should be on a login node. Heavy computations should not run here directly; they should be submitted to Slurm.

Now inspect the available partitions:

```bash
sinfo
echo "\"$SINFO_FORMAT\""
```

For a different detail:

```bash
SINFO_FORMAT="%.16P %.14F %.14C %.16L %.14l %.40G %.5D %N" sinfo --clusters=all
```

This shows partition names, availability, time limits, node counts, and node states. It helps answer the question: *where can my job run?*

Also check whether you already have jobs in the queue, as well as running and pending jobs on the GPU cluster:

```bash
squeue -u $USER
squeue -M gmerlin7 -t R
squeue -M gmerlin7 -t PD
echo "\"$SQUEUE_FORMAT\""
```

At this point, the important concept is that Slurm sees the cluster as a set of partitions and nodes. Users submit jobs to partitions, and Slurm decides when and where the job can run.

### Understanding the Slurm resource model

A batch script normally contains two distinct parts:

1. `#SBATCH` options request a **job allocation**.
2. Commands such as `srun` create **job steps** inside that allocation.

Consider:

```bash
#SBATCH --nodes=1
#SBATCH --ntasks=2
#SBATCH --cpus-per-task=4
#SBATCH --mem-per-cpu=512M
```

This requests:

| Resource | Amount |
|:---|---:|
| Nodes | 1 |
| Tasks | 2 |
| CPUs per task | 4 |
| Total allocated CPUs | 8 |
| Memory per allocated CPU | 512 MiB |
| Approximate total requested memory | 4096 MiB |

The key calculation is:

```text
total CPUs = ntasks × cpus-per-task
```

For `--mem-per-cpu`:

```text
total memory = allocated CPUs × memory per CPU
```

On a heterogeneous or multi-node allocation, always verify the exact result with Slurm rather than relying only on mental arithmetic.

#### Important distinctions

- A **task** is normally one process launched by `srun`.
- A **CPU** in Slurm generally means an allocatable processing unit according to the cluster configuration.
- `--cpus-per-task=4` reserves four CPUs for each task, but it does not automatically make the application use four threads.
- `OMP_NUM_THREADS=4` tells this particular workload to create four OpenMP threads per task.
- `--mem` requests memory per node.
- `--mem-per-cpu` scales the memory request with the number of allocated CPUs.

#### Understanding the resource request

Before submitting jobs, it is important to understand the most common Slurm resource options. A Slurm job request describes the *shape* of the resources needed by the application: how many nodes, how many tasks, how many CPU cores, how much memory, how much time, and, when relevant, how many GPUs.

On Merlin7, users should also be aware that there are two Slurm clusters:

- `merlin7` for CPU workloads
- `gmerlin7` for GPU workloads

The target Slurm cluster can be selected with the `--clusters` option, or with the equivalent short option `-M`.

<table>
  <thead>
    <tr>
      <th>Option</th>
      <th>Meaning</th>
      <th>Typical use</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>--clusters</code> / <code>-M</code></td>
      <td>Selects the Slurm cluster to which the command or job applies.</td>
      <td>
        Use <code>--clusters=merlin7</code> for CPU jobs and
        <code>--clusters=gmerlin7</code> for GPU jobs.
      </td>
    </tr>
    <tr>
      <td><code>--partition</code></td>
      <td>Selects the partition where the job should run.</td>
      <td>
        Used to select a class of resources or policy, for example CPU, GPU,
        short, long, interactive, or project-specific partitions.
      </td>
    </tr>
    <tr>
      <td><code>--reservation</code></td>
      <td>Requests nodes from an existing Slurm reservation.</td>
      <td>
        Can be used on some systems to select an active reservation which
        contains reserved resources on specific nodes. Reservations are
        often created by administrators or users with granted privileges.
      </td>
    </tr>
    <tr>
      <td><code>--nodes</code></td>
      <td>Number of compute nodes requested.</td>
      <td>
        Use when the job must run on one or more physical nodes. Many beginner
        jobs should start with <code>--nodes=1</code>.
      </td>
    </tr>
    <tr>
      <td><code>--hint</code></td>
      <td>Bind tasks according to application hints.</td>
      <td>
        Defines how tasks are bound to the allocated hardware resources.
        Available options include <code>compute_bound</code>, <code>memory_bound</code>, 
        <code>multithread</code> and <code>nomultithread</code>
      </td>
    </tr>
    <tr>
      <td><code>--ntasks</code></td>
      <td>Number of Slurm tasks to start.</td>
      <td>
        Often used for MPI ranks or independent parallel processes.
      </td>
    </tr>
    <tr>
      <td><code>--ntasks-per-node</code></td>
      <td>Controls how many tasks are placed on each node.</td>
      <td>
        Useful for multi-node jobs where the task distribution across nodes
        matters.
      </td>
    </tr>
    <tr>
      <td><code>--cpus-per-task</code></td>
      <td>Number of CPU cores assigned to each task.</td>
      <td>
        Used for threaded applications, OpenMP, Python multiprocessing, or any
        program where one process can use multiple CPU cores.
      </td>
    </tr>
    <tr>
      <td><code>--mem</code></td>
      <td>Memory requested per node.</td>
      <td>
        Use when the job needs a fixed amount of memory for the node allocation.
      </td>
    </tr>
    <tr>
      <td><code>--mem-per-cpu</code></td>
      <td>Memory requested per allocated CPU.</td>
      <td>
        Useful when the memory request should scale with the number of allocated
        CPU cores.
      </td>
    </tr>
    <tr>
      <td><code>--mem-per-gpu</code></td>
      <td>Memory requested per allocated GPU.</td>
      <td>
        Useful for GPU jobs when the memory request should scale with the number
        of GPUs. Do not combine it with <code>--mem</code> or
        <code>--mem-per-cpu</code>.
      </td>
    </tr>
    <tr>
      <td><code>--gpus</code></td>
      <td>Total number of GPUs requested for the job.</td>
      <td>
        Simple GPU request, typically used on <code>gmerlin7</code>.
      </td>
    </tr>
    <tr>
      <td><code>--gpus-per-node</code></td>
      <td>Number of GPUs requested on each allocated node.</td>
      <td>
        Useful when the job spans multiple GPU nodes and needs a fixed number
        of GPUs per node.
      </td>
    </tr>
    <tr>
      <td><code>--gpus-per-task</code></td>
      <td>Number of GPUs assigned per Slurm task.</td>
      <td>
        Useful when each task should receive one or more GPUs, for example in
        multi-process GPU workloads.
      </td>
    </tr>
    <tr>
      <td><code>--cpus-per-gpu</code></td>
      <td>Number of CPU cores requested per allocated GPU.</td>
      <td>
        Useful for GPU jobs that need CPU cores to feed each GPU efficiently.
      </td>
    </tr>
    <tr>
      <td><code>--gres=gpu:&lt;N&gt;</code></td>
      <td>Requests GPUs through Slurm generic resources.</td>
      <td>
        Common on many Slurm systems. Depending on the site configuration,
        this may be used instead of, or alongside, <code>--gpus</code>.
      </td>
    </tr>
    <tr>
      <td><code>--time</code></td>
      <td>Maximum runtime of the job.</td>
      <td>
        Helps Slurm schedule the job and defines when the job will be stopped
        if it exceeds its limit.
      </td>
    </tr>
  </tbody>
</table>

A common source of confusion is the difference between tasks and CPUs per task:

* A job with `--ntasks=4 --cpus-per-task=1` means: *start four separate tasks, each with one CPUs.* This is typical for MPI-style jobs or independent parallel processes.
* A job with `--ntasks=1 --cpus-per-task=4` means *start one task, but give it four CPUs.* This is typical for threaded applications. The application must actually use those threads; otherwise the extra CPUs are allocated but mostly idle.

For GPU jobs, the same idea still applies: CPU tasks and GPU resources are related, but they are not the same thing. For example, a job may request one task, several CPU threads, and one GPU: `--clusters=gmerlin7 --ntasks=1 --cpus-per-task=8 --gpus=1`. This means: *run one task, give it eight CPUs, and allocate one GPU.* This is a common shape for a single-process GPU application.

Another common source of confusion is <code>--hint=nomultithread</code>. It is often useful for pure MPI or compute-bound applications because it assigns only one task to each physical core instead of using sibling hardware threads. However, the full core is still allocated and accounted for. On systems with two threads per core, <code>sacct</code> may report both allocated CPUs, while <code>seff</code> may show around 50% CPU efficiency. This does not necessarily indicate poor performance: the job may still finish faster. The effect should be verified through benchmarking.

Finally, memory options should be chosen carefully. In general, use only one of: `--mem`, `--mem-per-cpu` or `--mem-per-gpu`:
* Use `--mem` when the job needs a fixed amount of memory per node.
* Use `--mem-per-cpu` when memory should scale with the number of CPU cores. On Merlin7, we have a default of `2912M` for the GPU cluster and `1888MB` for the CPU cluster
* Use `--mem-per-gpu` when memory should scale with the number of GPUs.

The main idea is that a Slurm resource request should describe what the application can really use. Requesting more CPUs, memory, nodes, or GPUs does not automatically make a job faster. It may only make the job harder to schedule if the application cannot use those resources efficiently.

## Exercises

### Exercise 1: One task, one CPU

Inspect the script:

```bash
cat 01_single_task.batch
```

Before submission, predict:

| Question | Expected answer |
|---|---:|
| Nodes | 1 |
| Tasks | 1 |
| CPUs per task | 1 |
| Total CPUs | 1 |
| Program processes | 1 |
| Threads in the process | 1 |

Submit it:

```bash
jobid=$(sbatch --parsable 01_single_task.batch)
echo "$jobid"
squeue -M merlin7 -j "$jobid"
```

While it is pending or running:

```bash
scontrol show job "$jobid"
```

After completion:

```bash
sacct -j "$jobid" --units=M \
  --format=JobID,JobName%22,State,Elapsed,AllocNodes,AllocCPUS,ReqMem,MaxRSS,ExitCode

seff $jobid
```

Read the output:

```bash
less 01-single-task-${jobid}.out
```

#### Questions

1. Does `AllocCPUS` equal one?
2. Does the program report one MPI rank and one OpenMP thread?
3. How many CPU IDs appear in the affinity mask?
4. Which information comes from the job request, and which comes from actual runtime placement?

<div class="alert alert-block alert-success">
<p><b>Main Lesson:</b></p>This is the smallest useful example: one Slurm task consumes one allocated CPU. The program happens to call that process an MPI rank, but from the Slurm perspective it is simply one task launched by <code>srun</code>.
</div>

### Exercise 2: Tasks and CPUs per task

Inspect:

```bash
cat 02_tasks_and_cpus.batch
```

The request is:

```text
--ntasks=2
--cpus-per-task=4
```

Predict the allocation before submission:

| Item | Prediction |
|---|---:|
| Tasks | 2 |
| CPUs per task | 4 |
| Total allocated CPUs | 8 |
| Processes launched by `srun` | 2 |
| Threads per process | 4 |
| Total application threads | 8 |

Submit:

```bash
jobid=$(sbatch --parsable 02_tasks_and_cpus.batch)
squeue -M merlin7 -j "$jobid"
```

Inspect the allocation:

```bash
scontrol show job "$jobid" | egrep 'JobId=|NumNodes=|NumCPUs=|NumTasks=|CPUs/Task=|ReqTRES=|AllocTRES='
```

After completion:

```bash
sacct -j "$jobid" --units=M \
  --format=JobID,JobName%22,State,AllocCPUS,ReqCPUS,ReqMem,MaxRSS,Elapsed

seff $jobid
```

Condense the program output:

```bash
grep -E 'Host:|MPI rank ID:|OpenMP threads in rank:|Linux CPU ID:|CPU affinity mask:' \
  02-tasks-cpus-${jobid}.out
```

#### Questions

1. Why are there two processes but eight allocated CPUs?
2. Does every task receive four CPUs?
3. Are the four threads of a task placed inside that task's CPU set?
4. What would happen to resource usage if `OMP_NUM_THREADS` were not set?
5. Would changing `--cpus-per-task` alone make the application create more threads?

<div class="alert alert-block alert-success">
<p><b>Main Lesson:</b></p><code>--ntasks</code> controls how many task processes <code>srun</code> launches. <code>--cpus-per-task</code> controls how many CPUs Slurm reserves for each task. The application must still be configured to use those CPUs.
</div>

### Exercise 3: Nodes, tasks per node, and distribution

Inspect:

```bash
cat 03_nodes_and_distribution.batch
```

This job requests:

```text
--nodes=2
--ntasks=4
--ntasks-per-node=2
--cpus-per-task=2
```

Predict:

| Item | Prediction |
|---|---:|
| Nodes | 2 |
| Tasks | 4 |
| Tasks per node | 2 |
| CPUs per task | 2 |
| CPUs per node used by the job | 4 |
| Total allocated CPUs | 8 |

Submit and inspect:

```bash
jobid=$(sbatch --parsable 03_nodes_and_distribution.batch)
squeue -j "$jobid" -o '%.18i %.8T %.6D %.6C %R'
scontrol show job "$jobid"
```

After completion:

```bash
sacct -j "$jobid" --format=JobID,JobName%24,State,AllocNodes,AllocCPUS,Elapsed

seff $jobid
```

Check which hosts ran the tasks:

```bash
grep -E '=====|Host:|MPI rank ID:' 03-nodes-${jobid}.out
```

The script creates two job steps, one with block distribution and one with cyclic distribution.

#### Questions

1. Does each step still use the same two-node job allocation?
2. How many tasks run on each node?
3. Does `--distribution` request additional resources?
4. Is distribution a property of the allocation, the step, or both?
5. With only two tasks per node, is there a visible difference between block and cyclic placement?

<div class="alert alert-block alert-success">
<p><b>Main Lesson:</b></p>The batch job owns a fixed allocation. Individual <code>srun</code> commands can choose how their tasks are distributed inside that allocation, provided they do not exceed it.
</div>

### Exercise 4: Multiple job steps in one allocation

Inspect:

```bash
cat 04_job_steps.batch
```

The job reserves four tasks and two CPUs per task, but it runs several different `srun` steps:

1. one task using two CPUs;
2. two tasks using two CPUs each;
3. four tasks using two CPUs each.

Submit:

```bash
jobid=$(sbatch --parsable 04_job_steps.batch)
```

While it runs, try:

```bash
squeue --steps -j "$jobid"
sstat -j "${jobid}.batch" --format=JobID,AveCPU,AveRSS,MaxRSS
```

After completion:

```bash
sacct -j "$jobid" --units=M \
  --format=JobID,JobName%24,State,NTasks,AllocCPUS,Elapsed,MaxRSS

seff $jobid
```

#### Questions

1. Which row represents the allocation itself?
2. Which rows represent the batch step and the `srun` steps?
3. Can one modify the job step name? How?
4. Does a step with one task release the unused CPUs to other jobs?
5. Can a step request more tasks or CPUs than the job allocation contains?
6. Why can a job have a low average CPU utilization even when one step used its CPUs efficiently?

<div class="alert alert-block alert-success">
<p><b>Main Lesson:</b></p>Resources belong to the job allocation for the lifetime of the job. A smaller step may leave part of the allocation idle, but those resources generally remain reserved for that job.
</div>

### Exercise 5: `--mem-per-cpu` versus `--mem`

The script submits equivalent-looking CPU layouts using two different memory request models.

Inspect:

```bash
cat 05_memory_requests.bash
```

#### Case A: Memory per CPU

```text
--ntasks=2
--cpus-per-task=2
--mem-per-cpu=256M
```

Expected total:

```text
2 tasks × 2 CPUs/task × 256 MiB/CPU = 1024 MiB
```

#### Case B: Memory per node

```text
--ntasks=2
--cpus-per-task=2
--mem=1024M
```

This requests 1024 MiB on the allocated node, independently of the number of tasks.

#### Case C: Using more memory than the allocated one

```text
--ntasks=2
--cpus-per-task=2
--mem-per-cpu=256M
```

The application will attempt to allocate more memory than the requested one.

#### Submit the jobs

<div class="alert alert-block alert-warning">
<p><b>This is not a Slurm batch script.</b></p>Notice that <code>05_memory_requests.bash</code> is not a Slurm batch script, but a script based on <b>BASH</b>. Therefore, it has to run as a normal <code>bash</code> script.
</div>

Submit the three jobs using the helper script:

```bash
bash 05_memory_requests.bash
```

It prints the three job IDs. Inspect them with:

```bash
scontrol show job <jobid> | egrep 'JobId=|NumCPUs=|ReqTRES=|AllocTRES=|MinMemory'

sacct -j <jobid> --units=M \
  --format=JobID,JobName%22,State,AllocCPUS,ReqMem,MaxRSS,Elapsed

seff <jobid>
```

Also inspect the out

#### Questions

1. Which request automatically grows when `--cpus-per-task` is increased?
2. If the job moves from one node to two nodes, how does `--mem` behave?
3. Why can the program estimate memory per task for `--mem-per-cpu`, but not reliably for `--mem`?
4. Is `MaxRSS` expected to equal the requested memory?
5. Why the third job has failed?

<div class="alert alert-block alert-success">
<p><b>Main Lesson:</b></p>A memory request is a limit and scheduling request, not a requirement that the application consume all of it. Request enough memory for the workload, but do not treat unused requested memory as free.
</div>

### Exercise 6: Detecting resource mismatch

Inspect:

```bash
cat 06_resource_mismatch.batch
```

The script demonstrates three common situations inside otherwise valid allocations.

#### Case A: CPUs requested but not used

The task receives four CPUs, but the program is configured with one thread.

**Expected consequence:** the job reserves four CPUs while the workload uses approximately one CPU.

#### Case B: matching request and use

The task receives four CPUs and creates four threads.

**Expected consequence:** the software has the opportunity to use all four CPUs.

#### Case C: more threads than allocated CPUs

The task receives two CPUs but creates four threads.

**Expected consequence:** multiple threads compete for the allocated CPUs. This is thread oversubscription inside the task's cpuset.

#### Submit the jobs

Submit the job:

```bash
jobid=$(sbatch --parsable 06_resource_mismatch.batch)
```

After completion:

```bash
sacct -j "$jobid" --format=JobID,JobName%26,State,AllocCPUS,Elapsed,TotalCPU,AveCPU

seff $jobid
```

Inspect affinity masks and thread counts:

```bash
grep -E '=====|OpenMP threads in rank:|OpenMP thread ID|Linux CPU ID|CPU affinity mask' \
  06-mismatch-${jobid}.out
```

#### Questions

1. In Case A, which resource is wasted?
2. In Case C, why does creating four threads not create four Slurm CPUs?
3. Which case is most likely to provide predictable performance?
4. How could CPU efficiency reveal Case A after completion?
5. [Optional] Would oversubscription allow the application to escape its allocated CPU set?

<div class="alert alert-block alert-success">
<p><b>Main Lesson:</b></p>The resource request and application configuration must agree. Slurm reserves resources; the application decides whether and how to use them.
</div>

### Exercise 7: NUMA memory binding

This exercise changes **where a task may allocate memory**, not how much memory Slurm reserves for the job. Both steps run inside the same allocation and use the same CPU and memory request:

```text
--nodes=1
--ntasks=2
--cpus-per-task=2
--mem-per-cpu=512M
```

Inspect the new script:

```bash
cat 07_memory_binding.batch
```

The script runs two sequential job steps.

#### Case A: no explicit memory binding

```bash
srun --cpu-bind=verbose,cores \
     --mem-bind=none \
     ./mpi_omp_affinity 128
```

`none` is Slurm's default memory-binding mode. Slurm does not establish a NUMA binding policy for the tasks. Linux first-touch placement may still put pages close to the CPU that first writes them, but the task is normally permitted to allocate from all NUMA nodes allowed by the job's cgroup or cpuset.

#### Case B: local memory binding

```bash
srun --cpu-bind=verbose,cores \
     --mem-bind=verbose,local \
     ./mpi_omp_affinity 128
```

`local` asks Slurm to bind each task to memory local to the processor on which that task runs. The program should normally report a narrower `Policy NUMA nodes` set, often one NUMA node per task when each task's CPU set lies inside one NUMA domain.

#### Submit the jobs

Submit the job:

```bash
jobid=$(sbatch --parsable 07_memory_binding.batch)
echo "$jobid"
```

Compare only the relevant lines:

```bash
grep -E '=====|Host:|MPI rank ID:|Linux CPU ID:|NUMA node OS index|Memory policy:|Policy NUMA nodes:|Physical NUMA location:' \
  07-memory-binding-${jobid}.out

# or
less 07-memory-binding-${jobid}.out
```

You can also inspect the step-level environment that Slurm creates:

```bash
sacct -j "$jobid" --format=JobID,JobName%24,State,NTasks,AllocCPUS,ReqMem

seff $jobid
```

#### Questions

1. Does changing `--mem-bind` change `AllocCPUS` or `ReqMem`?
2. With `--mem-bind=none`, which NUMA nodes appear under `Policy NUMA nodes`?
3. With `--mem-bind=local`, is the policy node set reduced to the NUMA node local to the task's CPUs?
4. Why can `Physical NUMA location` look local even when `--mem-bind=none` is used?
5. Is `--mem-bind=local` a memory-capacity request or a memory-placement policy?
6. What could happen if a task exhausts the free memory on its locally bound NUMA node?

<div class="alert alert-block alert-success">
<p><b>Main Lesson:</b></p><code>--mem</code> and <code>--mem-per-cpu</code> control the amount of memory requested and normally enforced. <code>--mem-bind</code> controls NUMA placement. With <code>none</code>, memory can come from any NUMA node permitted to the job; with <code>local</code>, Slurm restricts the task's policy to memory close to its CPU placement. The exact node mask depends on CPU placement, node topology, Slurm configuration, and cgroup constraints.
</div>